# J3-02 · RAG hybride sur Qdrant et sous-agents comme outils

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 3 · créneau 10h00–12h00 · ⏱️ ~2h (dont l'exercice 3.1, 45 min)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Indexer une base de connaissances dans Qdrant (dense + **BM25**, fusion **hybride**) · 2. Comparer recherche dense, lexicale et hybride · 3. Filtrer par **payload** (confidentialité, catégorie) · 4. Construire un spécialiste RAG qui **cite ses sources** · 5. Exposer des spécialistes **comme outils** d'un coordinateur, avec **isolation du contexte** |
| 🧱 **Fil rouge** | La recherche par mots-clés de J1 (qui renvoyait parfois le mauvais article) est remplacée par un vrai RAG |
| ✅ **Prérequis** | J3-01 ; Qdrant démarré ; modèles d'embeddings préchargés (`pull_models`) |
| 📦 **Versions** | `qdrant-client 1.19` · `langchain-qdrant 1.1` · `fastembed 0.9` · `sentence-transformers 6.1` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. La base de connaissances de NordTech

### [Théorie] 20 articles, et des métadonnées qui comptent autant que le texte

`data/kb/` contient 20 articles Markdown **fictifs**. Chaque en-tête porte des métadonnées qui deviendront le **payload** Qdrant : `categorie`, `equipe`, `visibilite` (`public` pour l'assistant N1, `interne` pour les procédures N2) et `maj`.

In [ ]:
from helpdesk_ops import kb

articles = kb.charger_articles()
print(f"{len(articles)} articles\n")
for a in articles[:3] + [a for a in articles if a.metadata["visibilite"] == "interne"]:
    m = a.metadata
    print(f"{m['id']}  {m['categorie']:<9} {m['equipe']:<9} {m['visibilite']:<8} {m['titre']}")
print("\n--- extrait de KB-008 ---\n" + next(a for a in articles if a.metadata["id"] == "KB-008").page_content[:420] + " …")

## 3. Indexer : trois choix vérifiés en préparant le cours

### [Théorie] Dense, lexical, hybride

| Recherche | Ce qu'elle capte | Ce qu'elle rate |
|---|---|---|
| **dense** (embeddings) | le **sens** : « tunnel sécurisé » ≈ « VPN » | les **termes exacts** rares : `0x80070005`, `srv-erp-02` |
| **lexicale** (BM25) | les **mots exacts**, les codes d'erreur | les reformulations, les synonymes |
| **hybride** | les deux, **fusionnés** (Reciprocal Rank Fusion) | — (au prix de deux index) |

Les trois choix de `helpdesk_ops.kb` :
1. **la collection est créée par notre code**, avec `Modifier.IDF` sur le vecteur creux : celle que crée `QdrantVectorStore.from_documents` n'a **pas** de modificateur, et BM25 perd sa pondération IDF (constaté dans la configuration de la collection) ;
2. **BM25 en français** (`language="french"`) : par défaut, fastembed racinise en anglais ; « les imprimantes sont bloquées » gardait alors « les » et « sont » ;
3. **identifiants stables** (`uuid5` de l'article et du morceau) : réindexer **remplace** au lieu de dupliquer.

> 💡 **Le « Tips » du Formateur :** les modèles d'embeddings (HF multilingue, BM25) se téléchargent **une fois** (`pull_models`), puis tout tourne hors ligne. Pendant la préparation, le premier chargement a pris 4 minutes, avec des coupures réseau : faites-le **la veille**.

In [ ]:
import time

debut = time.perf_counter()
n = kb.indexer(recreer=True)
print(f"{n} morceaux indexés en {time.perf_counter() - debut:.0f} s")

client = kb.client_qdrant()
config = client.get_collection(kb.nom_collection()).config.params
print("vecteur dense :", config.vectors.size, "dimensions,", config.vectors.distance)
print("vecteur creux :", config.sparse_vectors)

kb.indexer()   # réindexation : identifiants stables → aucun doublon
print("points après réindexation :", client.count(kb.nom_collection()).count)

**👀 Ce que vous devez observer :** 37 morceaux (un par section `##` des articles) ; le vecteur creux porte `modifier=<Modifier.IDF>` ; le nombre de points ne change pas après réindexation. Ouvrez aussi le **dashboard Qdrant** (`http://localhost:6333/dashboard`, clé `QDRANT_API_KEY` du `.env`) : collection `kb_nordtech`, payload de chaque point.

## 4. Comparer les trois recherches

In [ ]:
QUESTIONS = [
    "mon tunnel sécurisé ne fonctionne plus depuis l'hôtel",   # reformulation : le sens compte
    "0x80070005",                                              # code exact : le mot compte
    "l'ERP est en panne pour toute la compta",                 # les deux
]
print(f"{'question':<52} {'dense':<24} {'BM25':<24} {'hybride':<24}")
for q in QUESTIONS:
    res = {mode: [d.metadata["id"] for d in kb.rechercher(q, k=3, mode=mode)] for mode in ("dense", "sparse", "hybrid")}
    print(f"{q[:50]:<52} {', '.join(res['dense']):<24} {', '.join(res['sparse']):<24} {', '.join(res['hybrid']):<24}")

**👀 Ce que vous devez observer :** pour la reformulation, le dense trouve KB-001 (VPN) sans que le mot « VPN » soit écrit ; pour le code d'erreur, BM25 est sans ambiguïté ; l'hybride place le bon article en tête dans les trois cas. Remarquez que BM25 renvoie parfois **moins** de 3 résultats : aucun autre article ne contient les mots.

## 5. Filtres de payload : confidentialité et périmètre

### [Théorie] Le filtre s'applique **avant** le classement

Un filtre Qdrant restreint les points candidats **avant** la recherche. Deux usages :
- **confidentialité** (multi-tenant) : l'assistant N1 ne doit **jamais** recevoir les procédures internes N2 (redémarrer un serveur de production…). Ce n'est pas au LLM de « ne pas les citer » : elles ne doivent pas lui parvenir ;
- **périmètre** : un spécialiste Accès ne cherche que dans la catégorie `acces` (exercice 3.1).

Via LangChain, le payload est rangé sous `metadata` : on filtre sur `metadata.visibilite`. Les index de payload (`create_payload_index`) rendent ces filtres rapides à grande échelle.

In [ ]:
for q in ["procédure pour redémarrer un serveur de production", "alerte de supervision : 100 % de perte au ping"]:
    for public in (True, False):
        ids = [d.metadata["id"] for d in kb.rechercher(q, k=3, public_seulement=public)]
        print(f"{q[:50]:<52} public_seulement={public!s:<5} → {ids}")

print("\ncatégorie « acces » :", [d.metadata["id"] for d in kb.rechercher("je n'arrive plus à me connecter", categorie="acces")])
print("sans filtre         :", [d.metadata["id"] for d in kb.rechercher("je n'arrive plus à me connecter")])

**👀 Ce que vous devez observer :** avec le filtre (comportement par défaut de `kb.rechercher`), KB-019 et KB-018, procédures internes, n'apparaissent **jamais** ; sans filtre, elles arrivent en tête. Avec `categorie="acces"`, la question vague « je n'arrive plus à me connecter » trouve les procédures de comptes au lieu du VPN.

> 🛡️ **Sécurité :** le filtre est posé par le **code**, jamais demandé au LLM (« ne cite pas les documents internes ») : un modèle se laisse convaincre, un filtre non.

## 6. Un spécialiste RAG qui cite ses sources

### [Théorie] « Réponds à partir des extraits, et dis d'où ça vient »

Le spécialiste a **un** outil, `rechercher_procedures`, qui renvoie des extraits préfixés de leur référence `[KB-xxx]`. Sa consigne exige de citer les références, et de dire quand aucun extrait ne répond. On mesurera en J4 la **fidélité** de ses réponses aux sources.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool(parse_docstring=True)
def rechercher_procedures(question: str) -> str:
    """Cherche les procédures du help-desk NordTech les plus pertinentes pour un problème.

    Args:
        question: le problème décrit en une phrase, par exemple 'le VPN ne se connecte pas depuis l'hôtel'
    """
    return kb.formater(kb.rechercher(question, k=3))   # public seulement, par défaut


specialiste_kb = create_agent(
    llm, [rechercher_procedures], name="specialiste_kb",
    system_prompt="Tu es le spécialiste Procédures du help-desk NordTech. Utilise rechercher_procedures, puis réponds en "
                  "3 phrases maximum EN CITANT les références [KB-xxx] utilisées. Si aucun extrait ne répond, dis-le : n'invente rien.",
)
with propagate_attributes(session_id="j3-02-rag", tags=["jour-3", "rag"]):
    etat = await specialiste_kb.ainvoke({"messages": [("user", "Teams ne détecte plus mon micro")]},
                                        config={"callbacks": [langfuse_handler], "run_name": "specialiste-kb"})
print(etat["messages"][-1].content)
langfuse.flush()

**👀 Ce que vous devez observer :** la réponse cite `[KB-009]`. Dans Langfuse, la trace montre l'outil `rechercher_procedures` avec les **extraits exacts** reçus par le modèle : c'est là qu'on vérifie qu'il n'a rien ajouté.

> ⚠️ **Vu en préparant ce notebook :** la réponse citait bien `[KB-009]`… et ajoutait un chemin Windows détaillé (« Panneau de configuration > Système et sécurité > … ») qui n'est **pas** dans l'article. Citer une source ne prouve pas qu'on lui est fidèle.

## 7. Le pattern « sous-agents comme outils »

### [Théorie] Un spécialiste, vu du coordinateur, n'est qu'un outil

On enveloppe chaque spécialiste dans un outil `demander_<nom>(consigne)`. Le coordinateur décide **quoi** demander à **qui**, avec une consigne précise ; le spécialiste ne reçoit **que** cette consigne (pas la conversation, pas le nom de l'utilisateur) et rend un texte court. Le paramètre `config: RunnableConfig` propage les callbacks : les traces des spécialistes s'**imbriquent** dans celle du coordinateur.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

from helpdesk_ops.multi_agents import DESCRIPTIONS, PROMPT_COORDINATEUR, construire_specialistes


def comme_outil(nom: str, agent):
    async def deleguer(consigne: str, config: RunnableConfig) -> str:
        resultat = await agent.ainvoke({"messages": [HumanMessage(consigne)]},   # ← SEULE la consigne
                                       config={**config, "run_name": f"specialiste-{nom}"})
        return resultat["messages"][-1].content

    deleguer.__doc__ = f"""{DESCRIPTIONS[nom]}

        Args:
            consigne: la tâche précise confiée au spécialiste, avec les identifiants utiles (serveur, poste, compte)
        """
    return tool(f"demander_{nom}", parse_docstring=True)(deleguer)


specialistes = construire_specialistes(llm)
coordinateur = create_agent(llm, [comme_outil(nom, agent) for nom, agent in specialistes.items()],
                            system_prompt=PROMPT_COORDINATEUR, name="coordinateur")
print(PROMPT_COORDINATEUR)

In [ ]:
QUESTION = "Bonjour, ici Julie Martin. srv-erp-02 répond-il au ping ? Et comment je configure le VPN depuis l'hôtel ?"
debut = time.perf_counter()
with propagate_attributes(session_id="j3-02-coordinateur", tags=["jour-3", "sous-agents"]):
    etat = await coordinateur.ainvoke({"messages": [("user", QUESTION)]},
                                      config={"callbacks": [langfuse_handler], "run_name": "coordinateur", "recursion_limit": 25})
print(f"⏱ {time.perf_counter() - debut:.0f} s")
for m in etat["messages"]:
    for appel in getattr(m, "tool_calls", None) or []:
        print(f"→ {appel['name']}({appel['args']['consigne'][:90]!r})")
print("\n🤖", etat["messages"][-1].content)
langfuse.flush()

**👀 Ce que vous devez observer :**
- **deux** délégations (Diagnostic **et** KB) : là où le routeur de J3-01 n'en faisait qu'une ;
- les consignes rédigées par le coordinateur ne contiennent **pas** « Julie Martin » : l'isolation du contexte limite aussi la circulation des données personnelles ;
- dans Langfuse, la trace `coordinateur` contient les traces `specialiste-diagnostic` et `specialiste-kb` **imbriquées**, chacune avec ses propres outils.

> ⚠️ **Vu en préparant ce notebook :** avec une consigne qui demandait de « conserver les numéros de ticket », le coordinateur a **inventé** « Ticket : TKT-2025-0012 » alors qu'aucun ticket n'avait été créé. Le prompt actuel l'interdit explicitement, et l'exécution suivante n'en a plus inventé ; elle a en revanche traduit MFA par « authentification à facture », une erreur déjà vue en J1. Un prompt ne prouve rien : en J4, on **mesurera** ces hallucinations sur un dataset.

## 8. 🧪 Exercice 3.1 (45 min)

➡️ `exercices/enonces/Ex3_1_quatrieme_specialiste.ipynb` : ajouter un spécialiste **Accès** (annuaire + procédures filtrées sur la catégorie `acces`) et vérifier dans Langfuse que le coordinateur l'appelle à bon escient.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Indexer en hybride, proprement | collection créée avec `Modifier.IDF`, BM25 français, ids `uuid5` |
| Rechercher | `QdrantVectorStore(..., retrieval_mode=RetrievalMode.HYBRID)` |
| Filtrer par payload | `models.Filter(must=[FieldCondition(key="metadata.visibilite", …)])` |
| Construire un spécialiste RAG | `create_agent` + outil qui renvoie des extraits `[KB-xxx]` |
| Déléguer à des sous-agents | outil `demander_<nom>(consigne, config)` qui appelle `agent.ainvoke` |

### ✅ Quiz express

1. Pourquoi la recherche dense rate-t-elle `0x80070005` alors que BM25 le trouve ?
2. Pourquoi créer soi-même la collection plutôt que laisser `from_documents` le faire ?
3. Pourquoi filtrer la visibilité dans le code plutôt que demander au LLM de ne pas citer les documents internes ?
4. Que reçoit exactement un spécialiste appelé comme outil ?

<details>
<summary>Réponses</summary>

1. Un code rare n'a pas de « sens » que l'embedding saurait rapprocher ; BM25 compare les mots exacts.
2. Pour poser `Modifier.IDF` sur le vecteur creux (sans lui, BM25 perd sa pondération) et créer les index de payload.
3. Un LLM peut être convaincu de les citer ; un filtre empêche qu'ils lui parviennent.
4. Uniquement la consigne rédigée par le coordinateur, pas la conversation complète.
</details>